## Merging 2 main datasets

In [1]:
import sys
import numpy as np
import pandas as pd

Naming all files used & outputs

In [2]:
from pathlib import Path

IPEDS = Path("../data/merges/CS_PSEO_IPEDS_merged.csv")
BLS = Path("../data/merges/BLS_OEWS_ONET_merged.csv")
XW = Path("../data/raw/CIP_SOC_crosswalk/CIP2020_SOC2018_Crosswalk - CIP-SOC.csv")
OUT = Path("../data/merges/CS_PSEO_IPEDS_BLS_merged.csv")

In [3]:

ipeds = pd.read_csv(
    IPEDS,
    low_memory = False,
    dtype = {"OPEID6": str, "UNITID": str, "CIP_family": str, "CIP_series": str, "CIPCODE_str": str}
)
bls = pd.read_csv(
    BLS,
    low_memory = False,
    dtype = {"industry_code": str, "soc_code": str}
)
xw = pd.read_csv(
    XW,
    dtype = {"CIP2020Code": str, "SOC2018Code": str}
)

print("CS + PSEO + IPEDS:", ipeds.shape)
print("BLS + OEWS + O*NET:", bls.shape)
print("CIP-SOC crosswalk:", xw.shape)

CS + PSEO + IPEDS: (26678, 147)
BLS + OEWS + O*NET: (37799, 79)
CIP-SOC crosswalk: (6097, 4)


In [4]:
"""Merge CS_PSEO_IPEDS (program level, 4-digit CIP) with BLS_OEWS_ONET (occupation x industry)
via the NCES CIP2020 -> SOC2018 crosswalk.

Steps
1. Collapse BLS to one row per SOC (sum employment across industries, employment-weighted wages).
   NOTE: `employment`, `annual_*_wage`, `wage_spread`, `wage_progression_ratio` in the BLS file are
   constant within industry_code (industry-wide, all occupations), so they are NOT used here.
   Occupation-level fields are tot_emp / a_median / a_mean.
2. Trim crosswalk CIP to 4 digits, drop NO MATCH, dedupe CIP4-SOC pairs.
   Crosswalk SOCs missing from OEWS are mapped to OEWS aggregate codes (e.g. 13-1022 -> 13-1020).
3. Aggregate occupations to one row per CIP4 (employment-weighted), left-join onto IPEDS.
"""
# ipeds, bls, xw are loaded in the cell above; paths come from the naming cell
# OEWS values are strings with commas and "*"/"**" suppression flags -> numeric (NaN when suppressed)
for c in ["tot_emp", "a_median", "a_mean"]:
    bls[c] = pd.to_numeric(bls[c].astype(str).str.replace(",", "", regex=False), errors="coerce")


def wavg(v, w):
    m = v.notna() & w.notna() & (w > 0)
    if m.any():
        return np.average(v[m], weights=w[m])
    return v.mean()  # fall back to simple mean when employment is suppressed


# ---- 1. one row per SOC ---------------------------------------------------
onet_cols = ["unique_software_skills", "hot_technology_rate", "in_demand_rate",
             "technology_record_count", "average_importance", "importance_std"]
soc = (bls.groupby("soc_code")
          .apply(lambda g: pd.Series({
              "occ_title": g.occ_title.iloc[0],
              "occ_emp": g.tot_emp.sum(min_count=1),
              "occ_median_wage": wavg(g.a_median, g.tot_emp),
              "occ_mean_wage": wavg(g.a_mean, g.tot_emp),
              "n_industries": len(g),
              **{c: g[c].iloc[0] for c in onet_cols}}), include_groups=False)
          .reset_index())
soc["occ_w"] = soc.occ_emp.fillna(0)

# ---- 2. crosswalk to CIP4 ------------------------------------------------
xw = xw[xw.SOC2018Code != "99-9999"].copy()
xw["cip4"] = xw.CIP2020Code.str.zfill(7).str.replace(".", "", regex=False).str[:4]

bls_socs = set(soc.soc_code)


def to_oews(code):
    if code in bls_socs:
        return code, "exact"
    for cand in (code[:-1] + "0", code[:-2] + "00", code[:-1] + "1"):
        if cand in bls_socs:
            return cand, "aggregate"
    return None, "no_bls"


mapped = xw.SOC2018Code.map(to_oews)
xw["soc_code"] = mapped.str[0]
xw["soc_match"] = mapped.str[1]
xw_stats = xw.soc_match.value_counts()

bridge = (xw.dropna(subset=["soc_code"])[["cip4", "soc_code"]]
            .drop_duplicates()
            .merge(soc, on="soc_code", how="left"))

# ---- 3. one row per CIP4 -------------------------------------------------
def cip_agg(g):
    w = g.occ_w
    top = g.sort_values("occ_emp", ascending=False).occ_title.head(3)
    out = {
        "xw_n_occupations": len(g),
        "xw_occ_emp_total": g.occ_emp.sum(min_count=1),
        "xw_occ_median_wage": wavg(g.occ_median_wage, w),
        "xw_occ_mean_wage": wavg(g.occ_mean_wage, w),
        "xw_occ_median_wage_max": g.occ_median_wage.max(),
        "xw_top_occupations": "; ".join(top),
    }
    for c in onet_cols:
        out[f"xw_{c}"] = wavg(g[c], w)
    return pd.Series(out)


cip_feats = bridge.groupby("cip4").apply(cip_agg, include_groups=False).reset_index()

ipeds["cip4"] = ipeds.CIPCODE.astype(int).astype(str).str.zfill(4)
merged = ipeds.merge(cip_feats, on="cip4", how="left")
merged["has_soc_match"] = merged.xw_n_occupations.notna()

merged.to_csv(OUT, index=False)
bridge.to_csv(OUT.with_name(OUT.stem + "_cip_soc_bridge.csv"), index=False)

# ---- summary -------------------------------------------------------------
print("crosswalk SOC -> OEWS:", xw_stats.to_dict())
print("IPEDS rows:", len(ipeds), "-> merged rows:", len(merged))
print("row match rate:", round(merged.has_soc_match.mean(), 4))
print("CIP4 match rate:", round(ipeds.cip4.drop_duplicates().isin(cip_feats.cip4).mean(), 4),
      f"({ipeds.cip4.nunique()} distinct CIP4)")
um = (merged[~merged.has_soc_match].groupby(["cip4", "CIPDESC"]).size()
      .sort_values(ascending=False).head(15))
print("top unmatched CIP4 (rows):\n", um.to_string())

crosswalk SOC -> OEWS: {'exact': 5746, 'no_bls': 81, 'aggregate': 76}
IPEDS rows: 26678 -> merged rows: 26678
row match rate: 0.9834
CIP4 match rate: 0.9111 (360 distinct CIP4)
top unmatched CIP4 (rows):
 cip4  CIPDESC                                                                                 
3099  Multi/Interdisciplinary Studies, Other.                                                     67
5111  Health/Medical Preparatory Programs.                                                        42
5299  Business, Management, Marketing, and Related Support Services, Other.                       35
1199  Computer and Information Sciences and Support Services, Other.                              32
5199  Health Professions and Related Clinical Sciences, Other.                                    31
4399  Homeland Security, Law Enforcement, Firefighting and Related Protective Services, Other.    29
1599  Engineering/Engineering-Related Technologies/Technicians, Other.                        